![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--ag2-multi-agent--ag2-tutorial)

# Building Multi-Agent Systems with AG2

## Introduction to AG2

[AG2](https://ag2.ai/?utm_source=agents-towards-production&utm_medium=github&utm_campaign=tutorial) (formerly AutoGen) is an open-source framework for building multi-agent AI systems. With 500K+ monthly PyPI downloads and 4,300+ GitHub stars, AG2 provides a **conversation-centric approach** where agents collaborate through natural dialogue — no explicit routing graphs or transfer functions needed.

### Key Features:
- **Conversation-First Design:** Agents communicate through natural dialogue, making complex workflows intuitive
- **GroupChat Orchestration:** Automatic speaker selection using the LLM — add or remove agents without rewriting routing logic
- **Dual Tool Registration:** Clean separation between tool declaration and execution for sandboxing and approval flows
- **Flexible Termination:** Built-in conversation termination patterns for production safety

In this tutorial, we'll build up from a simple two-agent conversation to a fully orchestrated multi-agent system:
1. **Two-Agent Conversation:** AssistantAgent + UserProxyAgent fundamentals
2. **Tool Registration:** AG2's dual decorator pattern for function calling
3. **GroupChat:** Multi-agent orchestration with automatic speaker selection
4. **Production Patterns:** Termination strategies, human-in-the-loop, safety tips

## Setting Up Our Environment

First, let's install AG2 and set up our API keys. AG2 supports OpenAI, Azure, Bedrock, Ollama, and more.

In [ ]:
# Install AG2 with OpenAI support
!pip install -q ag2[openai] python-dotenv

### Loading API Keys

We'll load the OpenAI API key from environment variables. Create a `.env` file with your key:

```
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# AG2 imports — note: AG2 uses the 'autogen' namespace
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager, LLMConfig

# Configure LLM — AG2 supports OpenAI, Azure, Bedrock, Ollama, and more
llm_config = LLMConfig(
    {"model": "gpt-4o-mini"},
    temperature=0.7,
)

---

## Part 1: Two-Agent Conversation

AG2's fundamental pattern is a conversation between two agents:

- **AssistantAgent** — LLM-powered agent that reasons and generates responses
- **UserProxyAgent** — represents the human, can execute tools and provide input

This is the building block for all AG2 applications. Pass `llm_config` to each agent to configure which LLM it uses.

In [ ]:
assistant = AssistantAgent(
    name="Assistant",
    system_message=(
        "You are a helpful research assistant. Provide clear, well-structured "
        "answers with key facts and actionable insights."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",  # Automated mode (no manual input)
    max_consecutive_auto_reply=1,
    code_execution_config=False,
)

### Running the Conversation

The `initiate_chat` method starts a conversation. The UserProxy sends a message to the Assistant, which responds using the LLM.

In [ ]:
result = user_proxy.initiate_chat(
    assistant,
    message="What are the top 3 trends in AI agent development for 2025?",
)

### Key Concepts

| Parameter | Options | Behavior |
|-----------|---------|----------|
| `human_input_mode` | `"NEVER"` | Fully automated — no human input requested |
| | `"ALWAYS"` | Asks for human input every turn |
| | `"TERMINATE"` | Asks only when conversation ends |
| `max_consecutive_auto_reply` | Integer | Safety limit to prevent infinite loops |
| `code_execution_config` | `False` | Disables code execution (safer for production) |

---

## Part 2: Tool Registration

AG2 uses a **dual registration pattern** for tools:

1. **`register_for_llm`** — tells the LLM the tool exists (generates the schema)
2. **`register_for_execution`** — tells the agent how to actually execute it

This separation enables powerful patterns: the LLM decides **when** to call a tool, while the UserProxy controls **how** it's executed (with logging, sandboxing, or approval).

### Defining Tools

Tools are regular Python functions with `Annotated` type hints for parameter descriptions. AG2 automatically generates the tool schema from these annotations.

In [ ]:
from typing import Annotated


def search_knowledge_base(
    query: Annotated[str, "The search query to look up"],
    max_results: Annotated[int, "Maximum number of results to return"] = 3,
) -> str:
    """Search the internal knowledge base for relevant information."""
    # Simulated knowledge base for demo purposes
    knowledge = {
        "agent architectures": "Common patterns include ReAct, Plan-and-Execute, and Multi-Agent Debate.",
        "deployment": "Key considerations: containerization, API design, monitoring, and graceful degradation.",
        "evaluation": "Metrics include task completion rate, latency, cost per query, and user satisfaction.",
        "memory": "Types include short-term (conversation), long-term (vector DB), and episodic memory.",
        "safety": "Implement guardrails, content filtering, rate limiting, and human-in-the-loop for critical actions.",
    }
    results = []
    for topic, info in knowledge.items():
        if any(word in topic for word in query.lower().split()):
            results.append(f"**{topic}**: {info}")
    return "\n".join(results[:max_results]) if results else "No results found."


def generate_report(
    title: Annotated[str, "Report title"],
    content: Annotated[str, "Report content in markdown format"],
) -> str:
    """Generate a formatted report from the provided content."""
    report = f"# {title}\n\n{content}\n\n---\n*Generated by AG2 Multi-Agent System*"
    return report

### Registering Tools with Agents

The Researcher agent gets LLM awareness of the tools, while the Executor agent handles actual execution.

In [ ]:
researcher = AssistantAgent(
    name="Researcher",
    system_message=(
        "You are a research agent. Use the search_knowledge_base tool to find "
        "relevant information. Synthesize findings into clear summaries."
    ),
    llm_config=llm_config,
)
executor = UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config=False,
)

# Dual registration pattern
researcher.register_for_llm(description="Search knowledge base")(search_knowledge_base)
researcher.register_for_llm(description="Generate formatted report")(generate_report)
executor.register_for_execution()(search_knowledge_base)
executor.register_for_execution()(generate_report)

### Running the Tool-Enabled Conversation

The Researcher will decide when to call tools based on the task. The Executor handles tool execution and returns results.

In [ ]:
result = executor.initiate_chat(
    researcher,
    message="Research the best practices for deploying AI agents to production. "
            "Search for information about deployment, safety, and evaluation, "
            "then generate a report with your findings.",
)

### Why Dual Registration?

| Pattern | What it does | Why it matters |
|---------|-------------|----------------|
| `register_for_llm` | Generates tool schema for the LLM | LLM knows what tools exist and their parameters |
| `register_for_execution` | Wires up the actual function call | Execution can be sandboxed, logged, or require approval |

This separation means you can:
- Register a tool for LLM awareness but route execution through an approval workflow
- Log all tool calls without modifying tool code
- Execute tools in a sandbox while the LLM runs in a different environment

---

## Part 3: Multi-Agent GroupChat

GroupChat is AG2's flagship feature. Instead of building explicit routing graphs or writing transfer functions, you define agents and let the **GroupChatManager** automatically select the next speaker using the LLM.

This is how AG2 handles complex workflows with multiple specialists.

![AG2 GroupChat Architecture](assets/group-chat.png)

### Creating Specialist Agents

Each agent has a focused system message that describes its role. The GroupChatManager uses these descriptions to decide which agent should speak next.

In [ ]:
planner = AssistantAgent(
    name="Planner",
    system_message=(
        "You are a project planner. Break down complex tasks into clear steps. "
        "Identify which specialist should handle each step. "
        "Start by creating a plan, then coordinate the team."
    ),
    llm_config=llm_config,
)
researcher = AssistantAgent(
    name="Researcher",
    system_message=(
        "You are a research specialist. Gather information, analyze data, "
        "and provide factual findings. Always cite your reasoning."
    ),
    llm_config=llm_config,
)
writer = AssistantAgent(
    name="Writer",
    system_message=(
        "You are a technical writer. Take research findings and create "
        "clear, well-structured content for the target audience. "
        "Focus on actionable insights."
    ),
    llm_config=llm_config,
)
reviewer = AssistantAgent(
    name="Reviewer",
    system_message=(
        "You review content for accuracy, completeness, and clarity. "
        "Provide specific, constructive feedback. "
        "Say TERMINATE when the output meets quality standards."
    ),
    llm_config=llm_config,
)
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
)

### Setting Up GroupChat

The GroupChat takes a list of agents and a `speaker_selection_method`. With `"auto"`, the GroupChatManager uses the LLM to decide who speaks next based on the conversation context.

In [ ]:
groupchat = GroupChat(
    agents=[user_proxy, planner, researcher, writer, reviewer],
    messages=[],
    max_round=12,
    speaker_selection_method="auto",  # LLM-based speaker selection
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

### Running the Multi-Agent Conversation

The User sends a task to the GroupChatManager, which orchestrates the agents to collaboratively produce the output.

In [ ]:
result = user_proxy.initiate_chat(
    manager,
    message=(
        "Create a brief guide on 'Best Practices for Deploying AI Agents to Production'. "
        "Cover architecture, monitoring, safety, and scaling. "
        "Target audience: engineering teams building their first agent system."
    ),
)

### How GroupChat Works

1. **User sends a message** to the GroupChatManager
2. **GroupChatManager uses the LLM** to decide which agent should respond next (based on system messages, conversation history, and current context)
3. **Selected agent responds** — its message is added to the shared conversation
4. **GroupChatManager selects the next speaker** — the cycle continues
5. **Conversation ends** when `max_round` is reached or an agent says TERMINATE

### Speaker Selection Methods

| Method | Behavior | Best for |
|--------|----------|----------|
| `"auto"` | LLM decides the next speaker | Complex workflows with dynamic routing |
| `"round_robin"` | Agents take turns in order | Predictable pipelines |
| `"random"` | Random selection | Testing and exploration |
| `"manual"` | Human selects | Debugging and development |

### Production Tips

- **Set `max_round`** to prevent runaway conversations (8-15 rounds is typical)
- **Include TERMINATE** in one agent's system message for graceful exit
- **3-5 agents** is the sweet spot — more agents increases speaker selection complexity
- **Use descriptive agent names** — the GroupChatManager uses these for routing decisions

---

## Part 4: Production-Ready Patterns

### Termination Strategies

AG2 provides multiple ways to end conversations safely. Choosing the right strategy is critical for production deployments.

In [ ]:
# 1. Keyword termination (built-in)
agent_with_termination = AssistantAgent(
    name="Agent",
    system_message="Say TERMINATE when done.",
    llm_config=llm_config,
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),
)

# 2. Max rounds (via GroupChat)
# groupchat = GroupChat(agents=[...], max_round=10)

# 3. Custom termination function
def budget_check(msg):
    """Stop if we've exceeded our token budget."""
    return msg.get("content", "").count(" ") > 500  # Simple word count proxy

budget_agent = AssistantAgent(
    name="BudgetAgent",
    llm_config=llm_config,
    is_termination_msg=budget_check,
)

### Human-in-the-Loop

For critical decisions, AG2 supports human approval at different levels:

In [ ]:
# Human reviews every agent response before continuing
human_proxy = UserProxyAgent(
    name="HumanReviewer",
    human_input_mode="ALWAYS",  # Always asks for human input
    code_execution_config=False,
)

# Human reviews only at conversation end
approval_proxy = UserProxyAgent(
    name="ApprovalGate",
    human_input_mode="TERMINATE",  # Asks for input when agent says TERMINATE
    code_execution_config=False,
)

---

## Summary

| Concept | AG2 Pattern | When to Use |
|---------|------------|-------------|
| Simple task | AssistantAgent + UserProxyAgent | Single-purpose automation |
| Tool use | `register_for_llm` + `register_for_execution` | External API calls, data retrieval |
| Multi-agent | GroupChat + GroupChatManager | Complex workflows with specialists |
| Human oversight | `human_input_mode="ALWAYS"` | Critical decisions, compliance |
| Safety | `max_round` + `is_termination_msg` | All production deployments |

## Additional Considerations

### Limitations
- **Speaker selection cost**: `"auto"` mode makes an additional LLM call per turn to select the next speaker — adds latency and cost in high-throughput scenarios
- **Context window**: All agents share the full conversation history, which can exceed context limits in long conversations
- **Determinism**: LLM-based speaker selection is non-deterministic — same input may route differently across runs

### Possible Improvements
- Use `allowed_or_disallowed_speaker_transitions` to constrain which agents can follow which
- Implement conversation summarization for long-running GroupChats
- Add `select_speaker_prompt_template` for custom routing instructions
- Use `"round_robin"` for latency-sensitive pipelines where routing order is known

### Next Steps

- [AG2 Documentation](https://docs.ag2.ai/?utm_source=agents-towards-production&utm_medium=github&utm_campaign=tutorial) — full API reference, guides, and examples